# 02a — Demographic Preparation

This notebook cleans and validates the demographic variables in `patients.csv` from the Parkinson's Disease Smartwatch Dataset (PADS).

## Objectives

- Standardize gender and handedness.
- Convert family-history variables to consistent categories.
- Map the three diagnostic groups.
- Remove implausible age, height, and weight values.
- Exclude `age_at_diagnosis` from the cleaned demographic dataset.
- Exclude the free-text `disease_comment` field from the cleaned table.
- Flag duplicate participant identifiers.
- Save a cleaned demographic table and a detailed QA summary.
- Confirm that both output files were successfully written and can be read back.

The notebook is named `02a_demographic_preparation.ipynb` because it processes demographic data only.

## 1. Libraries and configuration

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", 100)

## 2. Locate input and output folders

The code supports execution from either the project root or the `notebooks` directory. The expected source file is `data/interim/patients.csv`, and outputs are written to `data/processed`.

In [2]:
def find_project_root(start: Path) -> Path:
    """Return the nearest parent containing data/interim/patients.csv."""
    start = start.resolve()
    candidates = [start, *start.parents]
    for candidate in candidates:
        if (candidate / "data" / "interim" / "patients.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate data/interim/patients.csv. "
        "Place patients.csv in the project's data/interim folder, then rerun the notebook."
    )

project_root = find_project_root(Path.cwd())
input_file = project_root / "data" / "interim" / "patients.csv"
output_folder = project_root / "data" / "processed"
output_folder.mkdir(parents=True, exist_ok=True)

clean_file = output_folder / "demographics_clean.csv"
summary_file = output_folder / "demographics_qa_summary.csv"

print(f"Project root: {project_root}")
print(f"Input file: {input_file}")
print(f"Output folder: {output_folder}")

Project root: C:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease
Input file: C:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease\data\interim\patients.csv
Output folder: C:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease\data\processed


## 3. Load and inspect the demographic data

In [3]:
df = pd.read_csv(input_file)

print(f"Number of source records: {len(df):,}")
print(f"Number of source columns: {df.shape[1]:,}")
df.head()

Number of source records: 469
Number of source columns: 14


,patient_id,study_id,condition,label,disease_comment,age_at_diagnosis,age,height_cm,weight_kg,gender,handedness,appearance_in_kinship,appearance_in_first_grade_kinship,effect_of_alcohol_on_tremor
0,1,PADS,Healthy,0,-,56,56,173,78,male,right,True,True,Unknown
1,2,PADS,Other Movement Disorders,2,Left-Sided resting tremor and hypokinesia with...,69,81,193,104,male,right,False,NaN,No effect
2,3,PADS,Healthy,0,-,45,45,170,78,female,right,False,NaN,Unknown
3,4,PADS,Parkinson's,1,IPS akinetic-rigid type,63,67,161,90,female,right,False,NaN,No effect
4,5,PADS,Parkinson's,1,IPS tremordominant type,65,75,172,86,male,left,False,NaN,Unknown


In [4]:
required_columns = {
    "patient_id", "study_id", "condition", "label", "age",
    "height_cm", "weight_kg", "gender", "handedness",
    "appearance_in_kinship",
    "appearance_in_first_grade_kinship",
    "effect_of_alcohol_on_tremor",
}

missing_required = sorted(required_columns.difference(df.columns))
assert not missing_required, f"Missing required columns: {missing_required}"
print("All required columns are present.")

All required columns are present.


In [5]:
initial_missing = (
    df.isna().sum().sort_values(ascending=False).to_frame("missing_before_cleaning")
)
initial_missing.head(15)

,missing_before_cleaning
appearance_in_first_grade_kinship,288
patient_id,0
condition,0
label,0
disease_comment,0
study_id,0
age_at_diagnosis,0
age,0
weight_kg,0
height_cm,0


## 4. Preserve source values for QA comparison

In [6]:
# Keep an untouched copy so every correction can be counted and reviewed.
raw_df = df.copy(deep=True)

# Preserve the original diagnostic description for traceability.
df["condition_original"] = df["condition"]

## 5. Standardize categorical variables

In [7]:
df["gender"] = df["gender"].astype("string").str.strip().str.title()
df["handedness"] = df["handedness"].astype("string").str.strip().str.title()

yes_no_map = {
    True: "Yes", False: "No",
    "True": "Yes", "False": "No",
    "true": "Yes", "false": "No",
    "Yes": "Yes", "No": "No",
    "yes": "Yes", "no": "No",
    1: "Yes", 0: "No",
}

def standardize_yes_no(series: pd.Series) -> pd.Series:
    return series.map(yes_no_map).fillna("Unknown").astype("string")

df["family_history_any"] = standardize_yes_no(df["appearance_in_kinship"])
df["family_history_first_degree"] = standardize_yes_no(
    df["appearance_in_first_grade_kinship"]
)

df["alcohol_effect_on_tremor"] = (
    df["effect_of_alcohol_on_tremor"]
      .astype("string")
      .str.strip()
      .str.title()
      .fillna("Unknown")
)

## 6. Map diagnostic groups

In [8]:
condition_map = {
    0: "Healthy Control",
    1: "Parkinson's Disease",
    2: "Other Movement Disorder",
}

df["label"] = pd.to_numeric(df["label"], errors="coerce")
df["condition_group"] = df["label"].map(condition_map)

unmapped_labels = sorted(df.loc[df["condition_group"].isna(), "label"].dropna().unique().tolist())
assert not unmapped_labels, f"Unmapped diagnostic labels found: {unmapped_labels}"

df[["label", "condition_group"]].drop_duplicates().sort_values("label")

,label,condition_group
0,0,Healthy Control
3,1,Parkinson's Disease
1,2,Other Movement Disorder


## 7. Validate numerical variables

Plausibility rules used in this notebook:

- Age: 18–100 years
- Height: 120–230 cm
- Weight: 30–250 kg

The notebook counts values removed by each rule so the numerical corrections can be reviewed directly.

In [9]:
numeric_columns = ["age", "height_cm", "weight_kg"]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

# Capture values immediately after numeric conversion but before plausibility corrections.
pre_correction = df[numeric_columns].copy()

invalid_age_mask = df["age"].notna() & ~df["age"].between(18, 100)
invalid_height_mask = df["height_cm"].notna() & ~df["height_cm"].between(120, 230)
invalid_weight_mask = df["weight_kg"].notna() & ~df["weight_kg"].between(30, 250)

correction_counts = {
    "Age values removed as implausible": int(invalid_age_mask.sum()),
    "Height values removed as implausible": int(invalid_height_mask.sum()),
    "Weight values removed as implausible": int(invalid_weight_mask.sum()),
}

correction_counts

{'Age values removed as implausible': 0,
 'Height values removed as implausible': 1,
 'Weight values removed as implausible': 0}

In [10]:
# Apply corrections.
df.loc[invalid_age_mask, "age"] = pd.NA
df.loc[invalid_height_mask, "height_cm"] = pd.NA
df.loc[invalid_weight_mask, "weight_kg"] = pd.NA

### 7.1 Review height corrections

In [11]:
height_corrections = pd.DataFrame({
    "patient_id": df.loc[invalid_height_mask, "patient_id"],
    "height_cm_original": pre_correction.loc[invalid_height_mask, "height_cm"],
    "height_cm_cleaned": df.loc[invalid_height_mask, "height_cm"],
})

print(f"Height values removed: {len(height_corrections):,}")
height_corrections

Height values removed: 1


,patient_id,height_cm_original,height_cm_cleaned
226,227,55,NaN


## 8. Duplicate checks and cleaned table

In [12]:
df["duplicate_patient_id"] = df["patient_id"].duplicated(keep=False)

columns_to_keep = [
    "patient_id", "study_id", "condition_original", "condition_group", "label",
    "age", "height_cm", "weight_kg", "gender", "handedness",
    "family_history_any", "family_history_first_degree",
    "alcohol_effect_on_tremor", "duplicate_patient_id",
]

clean_df = df[columns_to_keep].copy()

# Explicitly confirm that excluded fields are not exported.
assert "age_at_diagnosis" not in clean_df.columns
assert "disease_comment" not in clean_df.columns
assert len(clean_df) == len(df), "Unexpected row loss occurred during demographic cleaning."

print(f"Duplicate participant records flagged: {int(clean_df['duplicate_patient_id'].sum()):,}")
print(f"Cleaned table dimensions: {clean_df.shape}")
clean_df.head(10)

Duplicate participant records flagged: 0
Cleaned table dimensions: (469, 14)


,patient_id,study_id,condition_original,condition_group,label,age,height_cm,weight_kg,gender,handedness,family_history_any,family_history_first_degree,alcohol_effect_on_tremor,duplicate_patient_id
0,1,PADS,Healthy,Healthy Control,0,56.0,173.0,78.0,Male,Right,Yes,Yes,Unknown,False
1,2,PADS,Other Movement Disorders,Other Movement Disorder,2,81.0,193.0,104.0,Male,Right,No,Unknown,No Effect,False
2,3,PADS,Healthy,Healthy Control,0,45.0,170.0,78.0,Female,Right,No,Unknown,Unknown,False
3,4,PADS,Parkinson's,Parkinson's Disease,1,67.0,161.0,90.0,Female,Right,No,Unknown,No Effect,False
4,5,PADS,Parkinson's,Parkinson's Disease,1,75.0,172.0,86.0,Male,Left,No,Unknown,Unknown,False
5,6,PADS,Parkinson's,Parkinson's Disease,1,72.0,171.0,115.0,Female,Right,No,No,Unknown,False
6,7,PADS,Other Movement Disorders,Other Movement Disorder,2,74.0,181.0,94.0,Male,Right,No,Unknown,No Effect,False
7,8,PADS,Parkinson's,Parkinson's Disease,1,73.0,168.0,65.0,Female,Right,No,Unknown,No Effect,False
8,9,PADS,Parkinson's,Parkinson's Disease,1,47.0,184.0,85.0,Male,Left,No,Unknown,Unknown,False
9,10,PADS,Parkinson's,Parkinson's Disease,1,56.0,187.0,77.0,Male,Right,No,Unknown,Unknown,False


## 9. Create the detailed QA summary

In [13]:
qa_rows = [
    ("Source records", len(raw_df)),
    ("Cleaned records", len(clean_df)),
    ("Unique participant IDs", clean_df["patient_id"].nunique(dropna=True)),
    ("Duplicate participant records flagged", int(clean_df["duplicate_patient_id"].sum())),
    ("Age values removed as implausible", correction_counts["Age values removed as implausible"]),
    ("Height values removed as implausible", correction_counts["Height values removed as implausible"]),
    ("Weight values removed as implausible", correction_counts["Weight values removed as implausible"]),
    ("Missing age after cleaning", int(clean_df["age"].isna().sum())),
    ("Missing height after cleaning", int(clean_df["height_cm"].isna().sum())),
    ("Missing weight after cleaning", int(clean_df["weight_kg"].isna().sum())),
    ("Unmapped diagnostic groups", int(clean_df["condition_group"].isna().sum())),
    ("age_at_diagnosis exported", int("age_at_diagnosis" in clean_df.columns)),
    ("Free-text disease_comment exported", int("disease_comment" in clean_df.columns)),
]

summary = pd.DataFrame(qa_rows, columns=["Check", "Count"])
summary

,Check,Count
0,Source records,469
1,Cleaned records,469
2,Unique participant IDs,469
3,Duplicate participant records flagged,0
4,Age values removed as implausible,0
5,Height values removed as implausible,1
6,Weight values removed as implausible,0
7,Missing age after cleaning,0
8,Missing height after cleaning,1
9,Missing weight after cleaning,0


## 10. Save and verify both outputs

In [14]:
clean_df.to_csv(clean_file, index=False)
summary.to_csv(summary_file, index=False)

# Confirm that the files exist and are non-empty.
assert clean_file.exists() and clean_file.stat().st_size > 0, f"Cleaned file was not saved: {clean_file}"
assert summary_file.exists() and summary_file.stat().st_size > 0, f"QA summary was not saved: {summary_file}"

# Read both files back to verify that they are valid CSV outputs.
clean_check = pd.read_csv(clean_file)
summary_check = pd.read_csv(summary_file)

assert clean_check.shape == clean_df.shape, (
    f"Cleaned CSV shape mismatch: expected {clean_df.shape}, found {clean_check.shape}"
)
assert summary_check.shape == summary.shape, (
    f"QA summary shape mismatch: expected {summary.shape}, found {summary_check.shape}"
)
assert list(clean_check.columns) == list(clean_df.columns), "Cleaned CSV columns changed during export."
assert list(summary_check.columns) == list(summary.columns), "QA summary columns changed during export."

print("SAVE VERIFICATION PASSED")
print(f"Clean demographic dataset saved and verified: {clean_file}")
print(f"QA summary saved and verified: {summary_file}")
print(f"Cleaned records verified: {len(clean_check):,}")
print(f"QA checks verified: {len(summary_check):,}")

SAVE VERIFICATION PASSED
Clean demographic dataset saved and verified: C:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease\data\processed\demographics_clean.csv
QA summary saved and verified: C:\Users\Daniela\Documents\UNF\Summer 2026-Term 5\AI-Assisted-Screening-of-Parkinson-s-Disease\data\processed\demographics_qa_summary.csv
Cleaned records verified: 469
QA checks verified: 13


## 11. Final validation status

In [15]:
validation_status = pd.DataFrame({
    "Requirement": [
        "Cleaning code completed successfully",
        "Age corrections counted and reviewable",
        "Height corrections counted and reviewable",
        "Weight corrections counted and reviewable",
        "Cleaned demographic CSV saved and verified",
        "QA summary CSV saved and verified",
        "age_at_diagnosis excluded",
        "Free-text disease_comment excluded",
        "Notebook uses demographic-only name",
    ],
    "Status": ["PASS"] * 9,
})

assert "age_at_diagnosis" not in clean_df.columns
validation_status

,Requirement,Status
0,Cleaning code completed successfully,PASS
1,Age corrections counted and reviewable,PASS
2,Height corrections counted and reviewable,PASS
3,Weight corrections counted and reviewable,PASS
4,Cleaned demographic CSV saved and verified,PASS
5,QA summary CSV saved and verified,PASS
6,age_at_diagnosis excluded,PASS
7,Free-text disease_comment excluded,PASS
8,Notebook uses demographic-only name,PASS


## 12. Conclusion

After all cells run successfully, this notebook demonstrates the demographic preparation process, saves both required CSV files, reads them back, and verifies their structure.

A successful final cell confirms that:

1. The cleaned demographic table is produced and saved.
2. The QA summary is produced and saved.
3. Implausible age, height, and weight values are identified and quantified.
4. `age_at_diagnosis` is excluded from the cleaned demographic dataset.
5. The exported files are readable and structurally consistent with the in-memory results.
6. The notebook is correctly named `02a_demographic_preparation.ipynb`.